# 서비스셋 임베딩 run — 반경 안 상위 결과 (Colab GPU)

서비스셋 사례(쿼리 × 가정위치)마다 반경 안 가맹점만 놓고 임베딩 모델 상위 결과를 만듭니다. 판정 pool과 채점에 씁니다.
로컬 단계는 `scripts/build_service_pool.py`, 사례 묶음은 `SETS`(같은 파일)에 있습니다.

**반경별 상위 결과(2026-10-26 수정)**: 채점 반경(1km·2km)마다 그 안 상위 `run_depth`(50)개를 합쳐 저장합니다. 예전에는 2km 상위
50개만 남겨, 그 안에 1km 가게가 10개보다 적으면 1km 순위가 잘렸습니다.

1. 로컬: 묶음마다 `python scripts/build_service_pool.py export --set <묶음>` → `colab_upload/service_eval*/` 폴더들
   (`docs.csv`, `queries.csv`, `meta.json`)을 Drive `내 드라이브/store-search-ai/` 아래 같은 이름으로 올림(덮어쓰기)
2. 학습 모델(bge-m3 mj)이 Drive `store-search-ai/runs/finetune/bge_m3_ft_mj_20261014_0743/`에 있어야 함
3. Colab: 런타임 GPU(T4 가능) → 위에서부터 실행. 아래 `SETS`의 묶음을 차례로 돌려
   `store-search-ai/service_runs_v2/<run 폴더>/<모델>.csv`를 만듦(이미 있으면 건너뜀)
4. 로컬: Drive의 `service_runs_v2/` 안 폴더들을 `benchmark/service_ko_v1/` 아래 같은 이름 폴더(runs, runs_anchor, …)에 덮어씀
   → `python scripts/build_service_pool.py runs --set <묶음>` → `pool --set <묶음> --dry-run`

반경 안 후보 고르기·상위 k개(`# @cell runs`)는 로컬 `store_search_ai.data.service_pool`과 같은 규칙입니다
(`tests/test_service_dense_notebook.py`가 확인).

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/store-search-ai")
# 돌릴 묶음(Drive 입력 폴더 이름). 결과는 ROOT/service_runs_v2/<meta의 run_dir>/에 쓴다.
SETS = ["service_eval", "service_eval_anchor", "service_eval_extra", "service_eval_extra_b", "service_eval_anchor_extra",
        "service_eval_extra_a", "service_eval_anchor_extra_a"]
OUT_ROOT = ROOT / "service_runs_v2"
# 이름(run 파일 이름) → (모델 경로, query 지시문 이름). 지시문은 query 인코딩에만 붙는다(문서는 그대로).
# Qwen3(kse16)는 "query" 필수 — 모델에 저장된 지시문이 있지만 기본값이 꺼져 있어 넘기지 않으면 지시문 없이 인코딩된다.
# bge·Arctic은 None(기존 벤치마크 평가와 같은 조건, configs/models/*.yaml의 query_prompt_name).
MODELS = {
    "bge_m3_ft_mj_20261014_0743": ("/content/drive/MyDrive/store-search-ai/runs/finetune/bge_m3_ft_mj_20261014_0743", None),
    "bge_m3": ("BAAI/bge-m3", None),
    "kse16": ("<Drive의 qwen3_embedding_0_6b_ft_kse16_20261016_0012 폴더 경로>", "query"),
    "arcticLee": ("<Drive의 snowflake_arctic_embed_l_v2_ko_ft_lee_20261015_1434 폴더 경로>", None),
}
BATCH = 256

In [ ]:
# @cell runs
import numpy as np
import pandas as pd

EARTH_RADIUS_KM = 6371.0
RUN_COLUMNS = ["query_id", "doc_id", "rank", "score", "system"]


def haversine_km(lat, lon, lats, lons):
    a, b = np.radians(lat), np.radians(lon)
    la, lo = np.radians(lats), np.radians(lons)
    h = np.sin((la - a) / 2) ** 2 + np.cos(a) * np.cos(la) * np.sin((lo - b) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(h))


def radius_runs(docs, queries, doc_emb, q_emb, radius_km, depth, system, radii=None):
    """쿼리마다 반경 안 가맹점의 코사인 점수 상위 depth개. 동점은 docs 순서(로컬 top_k_run과 같은 규칙).
    radii(채점 반경)를 주면 반경마다 그 안 상위 depth개를 합친다(순위는 전체 점수 순)."""

    lats, lons = docs["latitude"].to_numpy(float), docs["longitude"].to_numpy(float)
    doc_ids = docs["doc_id"].to_numpy()
    cache, frames = {}, []
    for i, (qid, lat, lon) in enumerate(zip(queries["query_id"], queries["latitude"], queries["longitude"])):
        key = (float(lat), float(lon))
        if key not in cache:
            d = haversine_km(key[0], key[1], lats, lons)
            inside = np.flatnonzero(d <= radius_km)
            cache[key] = (inside, d[inside])
        cand, dist = cache[key]
        if len(cand) == 0:
            continue
        scores = doc_emb[cand] @ q_emb[i]
        order = np.lexsort((cand, -scores))
        if radii:
            keep = np.zeros(len(order), dtype=bool)
            keep[:depth] = True
            for r in radii:
                keep[np.flatnonzero(dist[order] <= r + 1e-9)[:depth]] = True
            order = order[keep]
        else:
            order = order[:depth]
        frames.append(pd.DataFrame({"query_id": qid, "doc_id": doc_ids[cand[order]], "rank": np.arange(1, len(order) + 1),
                                    "score": scores[order], "system": system}))
    return pd.concat(frames, ignore_index=True)[RUN_COLUMNS]

In [ ]:
import json


def load_set(name):
    d = ROOT / name
    docs = pd.read_csv(d / "docs.csv", encoding="utf-8-sig", dtype={"doc_id": str})
    queries = pd.read_csv(d / "queries.csv", encoding="utf-8-sig", dtype={"query_id": str})
    meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    return docs, queries, meta


for name in SETS:
    docs, queries, meta = load_set(name)
    print(f"{name}: 가맹점 {len(docs)}개, 사례 {len(queries)}개, 반경 {meta['judge_radius_km']}km, "
          f"반경별 상위 {meta['run_depth']}개 {meta.get('report_radii_km')} → {meta.get('run_dir')}")

In [ ]:
import gc

from sentence_transformers import SentenceTransformer

for name, (path, prompt) in MODELS.items():
    todo = []
    for s in SETS:
        _, _, meta = load_set(s)
        out = OUT_ROOT / meta["run_dir"] / f"{name}.csv"
        if out.exists():
            print(f"{out}: 이미 있음 — 건너뜀")
        else:
            todo.append((s, out))
    if not todo:
        continue
    model = SentenceTransformer(path, device="cuda", model_kwargs={"torch_dtype": torch.float16})
    if prompt is not None and prompt not in model.prompts:
        raise ValueError(f"{name}: 모델에 '{prompt}' 지시문이 없습니다 — {sorted(model.prompts)}")
    print(f"{name}: query 지시문 = {model.prompts.get(prompt) if prompt else '(없음)'!r}")
    for s, out in todo:
        docs, queries, meta = load_set(s)
        doc_emb = model.encode(docs["text"].fillna("").tolist(), batch_size=BATCH, normalize_embeddings=True,
                               show_progress_bar=True, convert_to_numpy=True).astype("float32")
        q_emb = model.encode(queries["query"].tolist(), prompt_name=prompt, batch_size=BATCH, normalize_embeddings=True,
                             convert_to_numpy=True).astype("float32")
        run = radius_runs(docs, queries, doc_emb, q_emb, meta["judge_radius_km"], meta["run_depth"], name,
                          meta.get("report_radii_km"))
        out.parent.mkdir(parents=True, exist_ok=True)
        run.to_csv(out, index=False, encoding="utf-8-sig", lineterminator="\n")
        print(f"[저장] {out} ({len(run)}행, 사례 {run['query_id'].nunique()}개)")
        del doc_emb
        gc.collect()
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
for s in SETS:
    _, queries, meta = load_set(s)
    for name in MODELS:
        run = pd.read_csv(OUT_ROOT / meta["run_dir"] / f"{name}.csv", encoding="utf-8-sig")
        print(s, name, len(run), "행 / 사례", run["query_id"].nunique(), "/ 사례당 평균",
              round(run.groupby("query_id").size().mean(), 1))